# 🍽️ Zomato Restaurant Data Analytics with AI
**Academic Internship Project**

This notebook performs end-to-end data analytics on the Zomato Restaurant dataset covering:
- Data loading and quality checks
- Data cleaning and preprocessing
- Exploratory Data Analysis (EDA)
- Location, rating, cost, cuisine and delivery analysis
- Correlation analysis
- Machine Learning: Restaurant rating prediction
- Key findings and business recommendations

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print('Libraries imported successfully.')

## 2. Load the Dataset
Loading the Zomato Restaurant CSV file from the project directory.

In [ ]:
df = pd.read_csv('Zomato Restaurant Dataset.csv', encoding='utf-8-sig')

print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(3)

## 3. Data Quality Checks
Inspecting data types, missing values, and duplicates before cleaning.

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
missing = df.isnull().sum()
empty  = (df == '').sum()
total_missing = missing + empty
print(total_missing[total_missing > 0])
print()
print(f'=== Duplicate Rows: {df.duplicated().sum()} ===')
print(f'=== Duplicate Restaurant IDs: {df["Restaurant ID"].duplicated().sum()} ===')

In [ ]:
df.describe(include='all').T

## 4. Data Cleaning & Preprocessing

Steps performed:
- Remove rows with missing `Cuisines`
- Map numeric `Country Code` to readable country names
- Encode Yes/No columns as binary integers
- Drop low-variance column `Switch to order menu` (almost all No)
- Remove unrated restaurants (rating = 0) for ML; keep them for general EDA
- Reset index after filtering

In [ ]:
# Country code → country name mapping (official Zomato country codes)
country_map = {
    1: 'India', 14: 'Australia', 30: 'Brazil', 37: 'Canada',
    94: 'Indonesia', 148: 'New Zealand', 162: 'Philippines',
    166: 'Qatar', 184: 'Singapore', 189: 'South Africa',
    191: 'Sri Lanka', 208: 'Turkey', 214: 'UAE',
    215: 'United Kingdom', 216: 'United States'
}

df['Country'] = df['Country Code'].map(country_map)

# Drop rows with missing Cuisines
df.dropna(subset=['Cuisines'], inplace=True)
df = df[df['Cuisines'].str.strip() != '']

# Drop low-variance column
df.drop(columns=['Switch to order menu'], inplace=True)

# Encode Yes/No columns to 1/0
yes_no_cols = ['Has Table booking', 'Has Online delivery', 'Is delivering now']
for col in yes_no_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Ensure numeric types
df['Aggregate rating'] = pd.to_numeric(df['Aggregate rating'], errors='coerce')
df['Votes']            = pd.to_numeric(df['Votes'],            errors='coerce')
df['Average Cost for two'] = pd.to_numeric(df['Average Cost for two'], errors='coerce')
df['Price range']      = pd.to_numeric(df['Price range'],      errors='coerce')

print(f'Clean dataset shape: {df.shape}')
print(f'Restaurants with 0.0 rating (unrated): {(df["Aggregate rating"] == 0).sum()}')
df.head(3)

## 5. Exploratory Data Analysis (EDA)
### 5.1 Restaurants per Country
India dominates with ~90% of all records; other countries provide global context.

In [ ]:
country_counts = df['Country'].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(country_counts.index, country_counts.values, color=sns.color_palette('muted', len(country_counts)))
ax.set_xlabel('Number of Restaurants')
ax.set_title('Number of Restaurants per Country')
for bar in bars:
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(country_counts.to_string())

### 5.2 Top 15 Cities by Restaurant Count
Cities with the highest restaurant presence on Zomato.

In [ ]:
top_cities = df['City'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top_cities.values, y=top_cities.index, palette='Blues_r', ax=ax)
ax.set_xlabel('Number of Restaurants')
ax.set_title('Top 15 Cities by Restaurant Count')
for i, v in enumerate(top_cities.values):
    ax.text(v + 5, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.show()

### 5.3 Rating Distribution
Distribution of aggregate ratings across all restaurants (including unrated ones).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of ratings
rated = df[df['Aggregate rating'] > 0]['Aggregate rating']
axes[0].hist(rated, bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Aggregate Ratings (Rated Only)')
axes[0].set_xlabel('Aggregate Rating')
axes[0].set_ylabel('Count')

# Rating text pie chart
rating_text_counts = df['Rating text'].value_counts()
colors = ['#2ecc71','#27ae60','#f39c12','#e74c3c','#95a5a6','#bdc3c7']
axes[1].pie(rating_text_counts.values, labels=rating_text_counts.index,
            autopct='%1.1f%%', startangle=140, colors=colors)
axes[1].set_title('Rating Category Distribution')

plt.tight_layout()
plt.show()

print('Rating text counts:')
print(rating_text_counts.to_string())

### 5.4 Price Range vs. Aggregate Rating
Do more expensive restaurants get higher ratings?

In [ ]:
rated_df = df[df['Aggregate rating'] > 0].copy()

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=rated_df, x='Price range', y='Aggregate rating',
            palette='YlOrRd', ax=ax)
ax.set_title('Rating Distribution by Price Range')
ax.set_xlabel('Price Range (1=Budget → 4=Premium)')
ax.set_ylabel('Aggregate Rating')
plt.tight_layout()
plt.show()

print(rated_df.groupby('Price range')['Aggregate rating'].describe().round(2))

### 5.5 Online Delivery & Table Booking Analysis
How many restaurants offer online delivery or table booking, and does it affect ratings?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Online delivery count
delivery_counts = df['Has Online delivery'].value_counts().rename({1:'Yes', 0:'No'})
axes[0].pie(delivery_counts, labels=delivery_counts.index, autopct='%1.1f%%',
            colors=['#3498db','#ecf0f1'], startangle=90)
axes[0].set_title('Online Delivery Available?')

# Table booking count
booking_counts = df['Has Table booking'].value_counts().rename({1:'Yes', 0:'No'})
axes[1].pie(booking_counts, labels=booking_counts.index, autopct='%1.1f%%',
            colors=['#e67e22','#ecf0f1'], startangle=90)
axes[1].set_title('Table Booking Available?')

# Rating comparison: delivery Yes vs No
delivery_label = rated_df['Has Online delivery'].map({1:'Has Delivery', 0:'No Delivery'})
sns.boxplot(data=rated_df, x=delivery_label, y='Aggregate rating',
            palette='Set2', ax=axes[2])
axes[2].set_title('Rating: Delivery vs No Delivery')
axes[2].set_xlabel('')
axes[2].set_ylabel('Aggregate Rating')

plt.tight_layout()
plt.show()

### 5.6 Top 20 Cuisines
Cuisines are multi-valued (comma-separated). Each cuisine type is counted individually.

In [ ]:
all_cuisines = []
for entry in df['Cuisines'].dropna():
    for c in entry.split(','):
        all_cuisines.append(c.strip())

cuisine_counts = pd.Series(Counter(all_cuisines)).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=cuisine_counts.values, y=cuisine_counts.index, palette='viridis', ax=ax)
ax.set_title('Top 20 Most Common Cuisines')
ax.set_xlabel('Number of Restaurants')
for i, v in enumerate(cuisine_counts.values):
    ax.text(v + 5, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.show()

### 5.7 Votes vs. Rating
Do restaurants with more votes tend to have higher ratings?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(rated_df['Votes'], rated_df['Aggregate rating'],
           alpha=0.25, s=10, color='steelblue')
ax.set_xlabel('Number of Votes')
ax.set_ylabel('Aggregate Rating')
ax.set_title('Votes vs. Aggregate Rating')
plt.tight_layout()
plt.show()

corr = rated_df['Votes'].corr(rated_df['Aggregate rating'])
print(f'Pearson correlation (Votes vs Rating): {corr:.3f}')

### 5.8 Online Delivery Rate by Country
Which countries have the highest proportion of restaurants with online delivery?

In [ ]:
delivery_by_country = df.groupby('Country')['Has Online delivery'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(9, 4))
delivery_by_country.plot(kind='barh', color='teal', ax=ax)
ax.set_xlabel('% Restaurants with Online Delivery')
ax.set_title('Online Delivery Rate by Country')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.show()

### 5.9 Average Rating by Country
Comparing average restaurant ratings across countries (rated restaurants only).

In [ ]:
avg_rating_country = (
    rated_df.groupby('Country')['Aggregate rating']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

fig, ax = plt.subplots(figsize=(9, 4))
avg_rating_country.plot(kind='bar', color=sns.color_palette('coolwarm', len(avg_rating_country)), ax=ax)
ax.set_ylabel('Average Aggregate Rating')
ax.set_title('Average Restaurant Rating by Country')
ax.set_xticklabels(avg_rating_country.index, rotation=45, ha='right')
ax.set_ylim(3.0, 4.5)
for i, v in enumerate(avg_rating_country.values):
    ax.text(i, v + 0.02, str(v), ha='center', fontsize=8)
plt.tight_layout()
plt.show()

### 5.10 Average Cost for Two by Country
> ⚠️ Note: Costs are in local currencies — this chart shows relative cost within each country's scale, not cross-country purchasing power comparisons.

In [ ]:
avg_cost = df.groupby('Country')['Average Cost for two'].mean().sort_values(ascending=False).round(0)

fig, ax = plt.subplots(figsize=(9, 4))
avg_cost.plot(kind='bar', color=sns.color_palette('Oranges_r', len(avg_cost)), ax=ax)
ax.set_ylabel('Average Cost for Two (Local Currency)')
ax.set_title('Average Cost for Two by Country (Local Currency Scale)')
ax.set_xticklabels(avg_cost.index, rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 5.11 Price Range Distribution
Most restaurants on Zomato fall in the budget to mid-range segment.

In [ ]:
price_dist = df['Price range'].value_counts().sort_index()
labels = ['1 - Budget', '2 - Mid-Range', '3 - Upscale', '4 - Premium']

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, price_dist.values, color=['#3498db','#2ecc71','#e67e22','#e74c3c'])
ax.set_ylabel('Number of Restaurants')
ax.set_title('Restaurant Count by Price Range')
for i, v in enumerate(price_dist.values):
    ax.text(i, v + 30, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 6. Correlation Analysis
Examining relationships between numeric features for rated restaurants.

In [ ]:
numeric_cols = ['Average Cost for two', 'Price range', 'Aggregate rating',
                'Votes', 'Has Table booking', 'Has Online delivery']

corr_matrix = rated_df[numeric_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            linewidths=0.5, square=True, ax=ax)
ax.set_title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

print('Correlation with Aggregate rating:')
print(corr_matrix['Aggregate rating'].drop('Aggregate rating').sort_values(ascending=False))

## 7. Machine Learning: Restaurant Rating Category Classification

**Objective:** Predict whether a restaurant is rated *Good/Very Good/Excellent* (High) or *Average/Poor* (Low) based on its features.

**Why this task is justified:**
- Clear binary target derived from `Rating text`
- Rich feature set: price, cost, location, delivery, booking, votes
- Business value: helps Zomato identify low-performing restaurants proactively

**Model used:** Random Forest Classifier (robust, interpretable, no feature scaling needed)

In [ ]:
# Use only rated restaurants
ml_df = rated_df.copy()

# Binary target: 1 = High (Good/Very Good/Excellent), 0 = Low (Average/Poor)
high_labels = ['Good', 'Very Good', 'Excellent']
ml_df['Rating_Class'] = ml_df['Rating text'].apply(lambda x: 1 if x in high_labels else 0)

print('Rating class distribution:')
print(ml_df['Rating_Class'].value_counts().rename({1:'High (Good+)', 0:'Low (Avg/Poor)'}))

# Encode Country as a numeric feature
le = LabelEncoder()
ml_df['Country_enc'] = le.fit_transform(ml_df['Country'])
ml_df['City_enc']    = le.fit_transform(ml_df['City'])

features = ['Price range', 'Average Cost for two', 'Votes',
            'Has Table booking', 'Has Online delivery',
            'Country_enc', 'City_enc']

X = ml_df[features]
y = ml_df['Rating_Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)
print(f'\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

In [ ]:
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Low (Avg/Poor)', 'High (Good+)']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Low', 'High'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')

# Feature importance
importances = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=True)
importances.plot(kind='barh', color='steelblue', ax=axes[1])
axes[1].set_title('Feature Importance (Random Forest)')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

## 8. Key Findings & Business Recommendations

### 📊 Key Findings

| # | Finding |
|---|---|
| 1 | **India dominates** the dataset with ~90.6% of all records (8,652 restaurants) |
| 2 | **22.5% of restaurants are unrated** — a significant engagement gap |
| 3 | **Budget restaurants (Price range 1)** are the most common (46.6% of all restaurants) |
| 4 | **Higher price range correlates with higher ratings** — premium restaurants consistently score better |
| 5 | **Online delivery is limited globally** — only India shows substantial online delivery adoption |
| 6 | **North Indian, Chinese, and Fast Food** are the three most listed cuisine types |
| 7 | **Votes correlate positively with rating** — more engaged restaurants tend to score higher |
| 8 | **The ML classifier achieves strong accuracy** in distinguishing high vs low rated restaurants, with `Votes` and `Price range` being the most predictive features |

### 💡 Business Recommendations

1. **Encourage unrated restaurants to collect feedback** — gamify or incentivize first reviews; 2,148 restaurants have zero engagement.
2. **Promote online delivery in under-penetrated countries** — most non-India countries have near-zero online delivery rates.
3. **Use the ML model as an early warning system** — proactively identify restaurants likely to score poorly and offer business support.
4. **Focus city-level marketing on New Delhi and Mumbai** — highest restaurant density and therefore largest user base opportunity.
5. **Table booking adoption is low overall** — promoting this feature could improve premium restaurant ratings and customer satisfaction.
6. **Diversify beyond North Indian cuisine** — it dominates, but data shows opportunity for under-represented cuisines to differentiate.

In [ ]:
print('=' * 55)
print('   ZOMATO DATA ANALYTICS PROJECT — SUMMARY')
print('=' * 55)
print(f'  Total restaurants analysed : {df.shape[0]:,}')
print(f'  Countries covered          : {df["Country"].nunique()}')
print(f'  Cities covered             : {df["City"].nunique():,}')
print(f'  Unique cuisine types       : {len(set(all_cuisines)):,}')
print(f'  Restaurants with delivery  : {df["Has Online delivery"].sum():,} ({df["Has Online delivery"].mean()*100:.1f}%)')
print(f'  Restaurants with booking   : {df["Has Table booking"].sum():,} ({df["Has Table booking"].mean()*100:.1f}%)')
print(f'  Avg rating (rated only)    : {rated_df["Aggregate rating"].mean():.2f} / 5.0')
print(f'  ML model accuracy (test)   : {(y_pred == y_test).mean()*100:.1f}%')
print('=' * 55)